# Classical Circuit Model: 2-bit × 2-bit multiplier

This notebook follows the same educational gate-level construction as the Python and Clojure projects. It is a **classical Boolean circuit**, not a quantum circuit.

Inputs are two unsigned bits per factor: `a = 2*a1 + a0`, `b = 2*b1 + b0`. The returned output is ordered `c3 c2 c1 c0`.

The construction computes four partial products with AND gates, reuses a shared AND term, and uses two XOR gates for the middle output bits. The count describes this lesson's construction, not a universal circuit-complexity minimum.

In [ ]:
from itertools import product


def multiply_2x2_bits(a1: int, a0: int, b1: int, b0: int) -> tuple[int, int, int, int]:
    """Return product bits (c3, c2, c1, c0) for four input bits."""
    for name, value in (("a1", a1), ("a0", a0), ("b1", b1), ("b0", b0)):
        if type(value) is not int or value not in (0, 1):
            raise ValueError(f"{name} must be the integer 0 or 1; got {value!r}")

    # Four partial-product AND gates.
    p0 = a0 & b0
    p1 = a0 & b1
    p2 = a1 & b0
    p3 = a1 & b1

    # Fifth AND gate: shared product/carry term.
    p4 = p0 & p3

    # Two XOR gates, plus direct wires.
    c0 = p0
    c1 = p1 ^ p2
    c2 = p4 ^ p3
    c3 = p4
    return (c3, c2, c1, c0)


def bits_to_int(bits: tuple[int, int, int, int]) -> int:
    c3, c2, c1, c0 = bits
    return (c3 << 3) | (c2 << 2) | (c1 << 1) | c0

## Exhaustive verification

There are only 4 × 4 = 16 possible input pairs. We can check every pair against ordinary integer multiplication and verify the shared carry identity `p1 AND p2 = p0 AND p3`.

In [ ]:
rows = []
for a, b in product(range(4), repeat=2):
    a1, a0 = (a >> 1) & 1, a & 1
    b1, b0 = (b >> 1) & 1, b & 1
    bits = multiply_2x2_bits(a1, a0, b1, b0)
    result = bits_to_int(bits)
    assert result == a * b, (a, b, result, a * b)

    p0, p1, p2, p3 = a0 & b0, a0 & b1, a1 & b0, a1 & b1
    assert (p1 & p2) == (p0 & p3), (a, b)
    rows.append((a, b, result, "".join(map(str, bits))))

print(" a  b | product | output (c3c2c1c0)")
for a, b, result, bit_string in rows:
    print(f" {a}  {b} |    {result:2}    |       {bit_string}")
print("PASS: all 16 input pairs and the shared carry identity passed.")

## What to notice

- A circuit is a composition of simple Boolean operations.
- Intermediate results can be wired to more than one later location, so a shared sub-expression need not be recomputed.
- `&` is AND for bit values, and `^` is XOR for bit values.
- The exhaustive test validates the implementation on all possible inputs for this 2-bit multiplier.